# 06b · Stage-2 locate & extract (GPU scaffold)

**Execution status:** 🔴 GPU SCAFFOLD — UNEXECUTED (runs on Colab, RUNBOOK.md)
**Plan reference:** PLAN §IV S2.0–S2.2 (RQ5) · THEORY §T8 · RUNBOOK session 6

The real counterpart to 06a: capture Qwen's residual-stream activations on the FULL/NEUTRAL dev generations and run the location toolkit on the true **d=3584** residuals (S2.0–S2.2). **No cell is executed here** — the hook logic is unit-tested against a tiny randomly-initialized Qwen2 stand-in (`tests/test_hooks.py`), never the 7B weights. Runs on Colab per `RUNBOOK.md` session 6.

| | |
|---|---|
| **Inputs** | FULL/NEUTRAL dev generations from P1 (reused as the extraction corpus) |
| **Outputs** | per-layer diff-in-means `v_ℓ`, probe AUC + selectivity, patching %-recovered; F5, F13 |
| **Runtime** | ≈ 2 A100-hours · ~40 GB (activation capture, chosen layers only) |

*Project 19 — Anatomy of a Design Skill. Governance: `CLAUDE.md`. Plan: `PLAN.md`. Derivations:
`THEORY.md`. Freeze: `PREREGISTRATION.md`. This notebook imports tested machinery from the `p19`
package and carries the narrative; it never re-implements logic that lives in `src/p19/`.*

> **Why unexecuted.** Loading Qwen-7B and capturing activations needs a GPU (compute policy). Every
> operation below is the *same* `p19` code notebook 06a validated on planted activations; here it is
> pointed at the real model and left unrun. The residual-stream hooks target `model.model.layers[i]`
> output `[0]`, handle tuple returns, and fire per decode step — all verified on the tiny stand-in.

In [ ]:
%matplotlib inline
# Bootstrap: locate the repo root (repo-relative — no hardcoded paths) and make p19 importable.
import sys, pathlib
_here = pathlib.Path.cwd()
_root = next((c for c in [_here, *_here.parents]
              if (c / "pyproject.toml").exists() and (c / "src" / "p19").exists()), _here)
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from p19 import REPO_ROOT
np.random.seed(0)                      # notebook-level seed; every generator also takes an explicit seed
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 120)
print("p19 ready · repo:", REPO_ROOT.name)

## S2.0 · The extraction corpus (reuse Stage-1 FULL/NEUTRAL dev generations)

No new generation: the 200 FULL and 200 NEUTRAL dev generations from P1 are re-run through a
**teacher-forced HF forward** (`[prompt ‖ response]`), capturing the mean-response residual per layer.

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: ~1 A100-hour · load 7B bf16 (~16 GB) + activation capture (chosen layers 11-20).
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from p19.config import model_config
from p19 import activations

mc = model_config()
tok = AutoTokenizer.from_pretrained(mc["model"]["name"])
model = AutoModelForCausalLM.from_pretrained(mc["model"]["name"], torch_dtype=torch.bfloat16,
                                             device_map="cuda").eval()
LAYERS = list(range(11, 21))                                 # S2.1 focus-band candidates

# build teacher-forced examples from the FULL/NEUTRAL dev HTML (response_start = len(prompt tokens))
examples = build_extraction_examples(tok, side="FULL") + build_extraction_examples(tok, side="NEUTRAL")
sides = activations.extract_corpus(model, examples, LAYERS, first_k=64)   # per-side per-layer mean vecs
activations.save_means(f"{os.environ['P19_RESULTS']}/activations/full_means.npz",
                       {l: np.mean(sides["FULL"][l], axis=0) for l in LAYERS})

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: seconds (numpy). Per-layer diff-in-means + the running-mean cosine-plateau check (F13).
from p19 import steering
v = {l: steering.diff_in_means(np.mean(sides["FULL"][l], 0), np.mean(sides["NEUTRAL"][l], 0))
     for l in LAYERS}
# stability: running means over the 200 pairs at each layer (PLAN S2.0 acceptance: Δcos < 0.01)
stab = activations.stability_report({l: [f - n for f, n in zip(sides["FULL"][l], sides["NEUTRAL"][l])]
                                     for l in LAYERS})
assert any(stab[l]["plateaued"] for l in LAYERS), "no layer plateaued -> extend to seeds 5-7 (PLAN S2.0)"
print("diff-in-means extracted; plateaued layers:", [l for l in LAYERS if stab[l]["plateaued"]])

## S2.1 · Probes, selectivity, and denoising patching

The same location logic as 06a, on real residuals: prompt-grouped CV probes with control-task
selectivity, and denoising patching with a **design-token logit-difference** proxy (e.g. a
non-`Inter` font-family token vs `Inter`; a non-purple hex digit vs the slop purple).

In [ ]:
# ═══ [GPU — COLAB ONLY — DO NOT RUN HERE] ═══
# Expected: ~0.5 A100-hour. Probes + patching → F5 + the RQ5 band (p19.probes, p19.patching).
from p19 import probes, patching, figures
means_by_layer = {l: np.vstack([sides["FULL"][l], sides["NEUTRAL"][l]]) for l in LAYERS}
labels = np.array([1]*len(sides["FULL"][LAYERS[0]]) + [0]*len(sides["NEUTRAL"][LAYERS[0]]))
groups = np.array(prompt_ids_full + prompt_ids_neutral)      # prompt id per example (no leak across folds)
stats = probes.probe_all_layers(means_by_layer, labels, groups, seed=0)

# denoising patch: patch the FULL-side clean vector into the NEUTRAL run; measure design-token logit diff
def metric(logits): return patching.design_token_logit_diff(logits, tok_plus=TOK_NON_INTER, tok_minus=TOK_INTER)
patch_frac = {l: patching.percent_recovered(
    patching.denoising_patch(model, corrupt_ids, v[l], l, position=-1, metric_fn=metric),
    m_corrupt, m_clean)/100 for l in LAYERS}

band = probes.choose_band(stats, patch_frac)
assert max(stats[l]["auc"] for l in LAYERS) >= 0.80, "no mid-band probe AUC >= 0.80 (RQ5 diffuse)"
figures.localization_plot(list(LAYERS), [stats[l]["auc"] for l in LAYERS],
                          [stats[l]["selectivity"] for l in LAYERS],
                          [patch_frac[l] for l in LAYERS], band=band["band"])
print("RQ5 band =", band["band"])

## Acceptance (before the sweep, RUNBOOK session 6)

Cosine plateau reached at the focus layers (else extend seeds 5–7); a mid-band probe **AUC ≥ 0.80**
with **selectivity ≥ 0.15**. The extracted `v_ℓ` and the band feed the dev sweep in notebook 07b.

---
**Note on testing.** Every hook here is exercised in `tests/test_hooks.py` and
`tests/test_patching_activations.py` against a 3-layer, hidden-64 Qwen2 stand-in with *known* forward
outputs — the capture/addition/ablation/patching math is verified without ever loading the 7B weights.